<a href="https://colab.research.google.com/github/jiyeonlee-2930/Math-in-AI/blob/main/cafeteria_sim_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 급식실 탑승 최적화 시뮬레이터

| 셀 | 내용 |
|---|---|
| 1 | 자바 파일 생성 (모델 + 엔진) |
| 2 | 컴파일 + 5전략 JSONL 데이터 생성 |
| 3 | Python 애니메이션 실행 |
| 4 | 실험자용 커스텀 이벤트 추가 예시 |

**`STRATEGY` 변수를 바꿔서 다른 전략을 시각화하세요.**
> `random` | `backtofront` | `outsidein` | `block` | `steffen`

In [1]:
%%shell
mkdir -p cafeteria

cat > cafeteria/TickEvent.java << 'JAVA_EOF'
package cafeteria;

import java.util.*;

// ================================================================
// TickEvent — 매 tick 에서 발생하는 이벤트의 기본 단위
//
// 실험자가 새 이벤트를 추가하려면 이 인터페이스를 구현하고
// TickEventRegistry.register()에 등록하면 됩니다.
// ================================================================
public interface TickEvent {
    /** 이벤트 이름 (로그·시각화에 사용) */
    String name();

    /** 이 이벤트가 완료되는 데 필요한 tick 수 */
    int durationTicks(Student student, CafeteriaConfig config);

    /** 이벤트 완료 시 호출되는 콜백 */
    default void onComplete(Student student, CafeteriaSimulator sim) {}
}

// ================================================================
// 내장 TickEvent 5가지
// ================================================================

/** 1. 복도 이동: 교실 → 급식실 입구 */
class HallwayMoveEvent implements TickEvent {
    @Override public String name() { return "HALLWAY_MOVE"; }
    @Override
    public int durationTicks(Student s, CafeteriaConfig cfg) {
        return cfg.hallwayMoveTicks;
    }
    @Override
    public void onComplete(Student s, CafeteriaSimulator sim) {
        s.advanceState();
    }
}

/** 2. 입구 대기: 급식실 입구에서 배식대 진입 대기 */
class EntranceWaitEvent implements TickEvent {
    @Override public String name() { return "ENTRANCE_WAIT"; }
    @Override
    public int durationTicks(Student s, CafeteriaConfig cfg) {
        // 배식 라인 수로 나눈 대기 시간
        return Math.max(1, cfg.entranceWaitTicks / cfg.serviceLines);
    }
    @Override
    public void onComplete(Student s, CafeteriaSimulator sim) {
        s.advanceState();
    }
}

/** 3. 배식 수령: 배식대에서 음식 받기 */
class FoodServiceEvent implements TickEvent {
    @Override public String name() { return "FOOD_SERVICE"; }
    @Override
    public int durationTicks(Student s, CafeteriaConfig cfg) {
        // 메뉴 수 × 1인당 배식 시간
        return cfg.menuItems * cfg.serviceTimePerItem + s.getExtraBaggageTicks();
    }
    @Override
    public void onComplete(Student s, CafeteriaSimulator sim) {
        s.advanceState();
    }
}

/** 4. 자리 이동: 배식 완료 후 지정 자리로 이동 */
class SeatMoveEvent implements TickEvent {
    @Override public String name() { return "SEAT_MOVE"; }
    @Override
    public int durationTicks(Student s, CafeteriaConfig cfg) {
        return cfg.seatMoveTicks + s.getInterferenceTicks(cfg);
    }
    @Override
    public void onComplete(Student s, CafeteriaSimulator sim) {
        s.advanceState();
    }
}

/** 5. 착석 완료: 자리에 앉는 동작 */
class SitDownEvent implements TickEvent {
    @Override public String name() { return "SIT_DOWN"; }
    @Override
    public int durationTicks(Student s, CafeteriaConfig cfg) {
        return cfg.sitDownTicks;
    }
    @Override
    public void onComplete(Student s, CafeteriaSimulator sim) {
        s.setState(Student.State.EATING);
        sim.onStudentSeated(s);
    }
}

// ================================================================
// TickEventRegistry — 실험자가 이벤트를 추가/교체할 수 있는 레지스트리
//
// 사용 예:
//   TickEventRegistry.register(new PaymentEvent());   // 결제 이벤트 추가
//   TickEventRegistry.replace("FOOD_SERVICE", new SlowServiceEvent()); // 교체
// ================================================================
class TickEventRegistry {
    private static final LinkedHashMap<String, TickEvent> registry = new LinkedHashMap<>();

    static {
        // 기본 5가지 이벤트 등록 (순서 = 진행 순서)
        register(new HallwayMoveEvent());
        register(new EntranceWaitEvent());
        register(new FoodServiceEvent());
        register(new SeatMoveEvent());
        register(new SitDownEvent());
    }

    public static void register(TickEvent event) {
        registry.put(event.name(), event);
    }

    public static void replace(String name, TickEvent event) {
        if (!registry.containsKey(name))
            throw new IllegalArgumentException("등록되지 않은 이벤트: " + name);
        registry.put(name, event);
    }

    public static void insertAfter(String afterName, TickEvent newEvent) {
        LinkedHashMap<String, TickEvent> newMap = new LinkedHashMap<>();
        for (Map.Entry<String, TickEvent> e : registry.entrySet()) {
            newMap.put(e.getKey(), e.getValue());
            if (e.getKey().equals(afterName)) {
                newMap.put(newEvent.name(), newEvent);
            }
        }
        registry.clear();
        registry.putAll(newMap);
    }

    public static List<TickEvent> getSequence() {
        return new ArrayList<>(registry.values());
    }

    public static List<String> getNames() {
        return new ArrayList<>(registry.keySet());
    }

    public static void printAll() {
        System.out.println("=== 등록된 TickEvent 순서 ===");
        int i = 1;
        for (String name : registry.keySet())
            System.out.printf("  %d. %s%n", i++, name);
    }
}

JAVA_EOF

cat > cafeteria/CafeteriaConfig.java << 'JAVA_EOF'
package cafeteria;

// ================================================================
// CafeteriaConfig — 급식실 시뮬레이션 전체 설정값
//
// 실험자가 이 값을 바꿔서 다양한 시나리오를 실험할 수 있습니다.
// ================================================================
public class CafeteriaConfig {

    // ── 학생 설정 ────────────────────────────────────────────────
    public int studentCount     = 30;   // 전체 학생 수
    public int classCount       = 1;    // 학급 수

    // ── 공간 설정 ────────────────────────────────────────────────
    public int tableRows        = 5;    // 식탁 행 수
    public int tableCols        = 6;    // 식탁 열 수 (한 행에 앉을 수)
    public int serviceLines     = 2;    // 배식 라인 수

    // ── tick 시간 설정 (1 tick ≈ 1초) ────────────────────────────
    public int hallwayMoveTicks     = 5;    // 복도 이동 시간
    public int entranceWaitTicks    = 4;    // 입구 대기 기본 시간
    public int menuItems            = 4;    // 배식 메뉴 수 (밥·국·반찬1·반찬2)
    public int serviceTimePerItem   = 2;    // 메뉴 1개 배식 시간
    public int seatMoveTicks        = 3;    // 자리 이동 시간
    public int sitDownTicks         = 1;    // 착석 동작 시간

    // ── 방해 요소 설정 ────────────────────────────────────────────
    public int interferencePerPerson = 2;   // 옆자리 간섭 1회당 지연 tick
    public double baggageRatio       = 0.2; // 짐(가방 등) 때문에 느린 학생 비율
    public int extraBaggageTicks     = 3;   // 짐 있는 학생의 추가 배식 시간

    // ── 입장 전략 설정 ────────────────────────────────────────────
    public String strategy = "random"; // random | backtofront | outsidein | block | steffen

    // ── 편의 메서드 ──────────────────────────────────────────────

    /** 총 좌석 수 */
    public int totalSeats() { return tableRows * tableCols; }

    /** 배식 1인당 평균 소요 tick */
    public int avgServiceTicks() { return menuItems * serviceTimePerItem; }

    @Override
    public String toString() {
        return String.format(
            "Config{students=%d, lines=%d, tables=%dx%d, " +
            "hallway=%d, service=%d×%d, strategy=%s}",
            studentCount, serviceLines, tableRows, tableCols,
            hallwayMoveTicks, menuItems, serviceTimePerItem, strategy
        );
    }

    // ── 빌더 패턴 (메서드 체이닝으로 설정) ──────────────────────

    public CafeteriaConfig students(int n)          { studentCount = n;          return this; }
    public CafeteriaConfig lines(int l)             { serviceLines = l;          return this; }
    public CafeteriaConfig tables(int r, int c)     { tableRows = r; tableCols = c; return this; }
    public CafeteriaConfig hallway(int t)           { hallwayMoveTicks = t;      return this; }
    public CafeteriaConfig menu(int items, int t)   { menuItems = items; serviceTimePerItem = t; return this; }
    public CafeteriaConfig strategy(String s)       { this.strategy = s;         return this; }
    public CafeteriaConfig baggage(double r, int t) { baggageRatio = r; extraBaggageTicks = t; return this; }
}

JAVA_EOF

cat > cafeteria/Student.java << 'JAVA_EOF'
package cafeteria;

import java.util.List;
import java.util.Random;

// ================================================================
// Student — 급식실 학생 한 명
// ================================================================
public class Student {

    public enum State {
        WAITING,            // 교실 대기
        HALLWAY_MOVE,       // 복도 이동 중
        ENTRANCE_WAIT,      // 입구 대기 중
        FOOD_SERVICE,       // 배식 받는 중
        SEAT_MOVE,          // 자리로 이동 중
        SIT_DOWN,           // 앉는 중
        EATING              // 착석 완료 (식사 중)
    }

    private static final Random RNG = new Random();

    private final int id;
    private final int targetRow;     // 배정된 자리 행
    private final int targetCol;     // 배정된 자리 열
    private final boolean hasBaggage; // 짐이 있는 학생

    private State state;
    private int ticksLeft;           // 현재 상태 남은 tick
    private int eventIndex;          // 현재 진행 중인 이벤트 인덱스
    private int waitTicks;           // 대기(막힘)로 소비된 tick 누적
    private int totalTicks;          // 이 학생의 총 소요 tick
    private int queuePosition;       // 배식 줄 위치 (시각화용)

    public Student(int id, int targetRow, int targetCol, CafeteriaConfig cfg) {
        this.id = id;
        this.targetRow = targetRow;
        this.targetCol = targetCol;
        this.hasBaggage = RNG.nextDouble() < cfg.baggageRatio;
        this.state = State.WAITING;
        this.ticksLeft = 0;
        this.eventIndex = 0;
        this.waitTicks = 0;
        this.totalTicks = 0;
        this.queuePosition = -1;
    }

    // ── 상태 진행 ────────────────────────────────────────────────

    /** 현재 이벤트의 tick을 1 소비. 완료 여부 반환 */
    public boolean tickDown() {
        totalTicks++;
        if (ticksLeft > 0) ticksLeft--;
        return ticksLeft == 0;
    }

    /** 다음 상태로 전환 */
    public void advanceState() {
        eventIndex++;
        List<String> names = TickEventRegistry.getNames();
        if (eventIndex < names.size()) {
            state = stateFromEventName(names.get(eventIndex));
        }
    }

    /** 현재 이벤트에 맞는 tick 설정 */
    public void initTicks(CafeteriaConfig cfg) {
        List<TickEvent> events = TickEventRegistry.getSequence();
        if (eventIndex < events.size()) {
            ticksLeft = events.get(eventIndex).durationTicks(this, cfg);
        }
    }

    public void setState(State s) { this.state = s; }
    public void addWaitTick()     { waitTicks++; totalTicks++; }

    // ── interference 계산 ────────────────────────────────────────

    public int getInterferenceTicks(CafeteriaConfig cfg) {
        // 이미 착석한 사람이 이동 경로를 막을 경우 (창가 자리 등)
        // 단순화: 열 번호가 0이나 마지막이면 간섭 가능성 있음
        int blockingCount = 0;
        if (targetCol == 0 || targetCol == tableCols(cfg) - 1) {
            blockingCount = RNG.nextInt(2); // 0 또는 1
        }
        return blockingCount * cfg.interferencePerPerson;
    }

    private int tableCols(CafeteriaConfig cfg) { return cfg.tableCols; }

    public int getExtraBaggageTicks() {
        return hasBaggage ? 3 : 0; // CafeteriaConfig.extraBaggageTicks와 연동
    }

    // ── 상태 조회 ────────────────────────────────────────────────

    public boolean isWaiting()      { return state == State.WAITING; }
    public boolean isEating()       { return state == State.EATING; }
    public boolean isDone()         { return state == State.EATING; }

    public int getId()              { return id; }
    public int getTargetRow()       { return targetRow; }
    public int getTargetCol()       { return targetCol; }
    public boolean hasBaggage()     { return hasBaggage; }
    public State getState()         { return state; }
    public int getTicksLeft()       { return ticksLeft; }
    public int getEventIndex()      { return eventIndex; }
    public int getWaitTicks()       { return waitTicks; }
    public int getTotalTicks()      { return totalTicks; }
    public int getQueuePosition()   { return queuePosition; }
    public void setQueuePosition(int p) { queuePosition = p; }

    // ── 유틸 ─────────────────────────────────────────────────────

    private State stateFromEventName(String name) {
        return switch (name) {
            case "HALLWAY_MOVE"   -> State.HALLWAY_MOVE;
            case "ENTRANCE_WAIT"  -> State.ENTRANCE_WAIT;
            case "FOOD_SERVICE"   -> State.FOOD_SERVICE;
            case "SEAT_MOVE"      -> State.SEAT_MOVE;
            case "SIT_DOWN"       -> State.SIT_DOWN;
            default               -> State.EATING;
        };
    }

    public int stateCode() {
        return switch (state) {
            case WAITING        -> 0;
            case HALLWAY_MOVE   -> 1;
            case ENTRANCE_WAIT  -> 2;
            case FOOD_SERVICE   -> 3;
            case SEAT_MOVE      -> 4;
            case SIT_DOWN       -> 5;
            case EATING         -> 6;
        };
    }

    @Override
    public String toString() {
        return String.format("S%d[%s→(%d,%d)%s]",
            id, state, targetRow, targetCol, hasBaggage ? "+bag" : "");
    }
}

JAVA_EOF

cat > cafeteria/CafeteriaSimulator.java << 'JAVA_EOF'
package cafeteria;

import java.util.*;

public class CafeteriaSimulator {

    private final CafeteriaConfig cfg;

    private final Deque<Student> waitingQueue;
    private final List<Student>  inHallway;
    private final List<Student>  atEntrance;
    private final List<Student>  atService;
    private final List<Student>  movingToSeat;
    private final List<Student>  sittingDown;
    private final List<Student>  seated;

    private final boolean[][] seatOccupied;

    private int totalTicks;
    private int bottleneckTicks;

    private final List<Map<String, Object>> snapshots;
    private final boolean recordSnapshots;

    // 이번 tick에 착석 완료된 학생들 (임시 버퍼)
    private final List<Student> justSeated = new ArrayList<>();

    public CafeteriaSimulator(CafeteriaConfig cfg, List<Student> queue, boolean recordSnapshots) {
        this.cfg = cfg;
        this.recordSnapshots = recordSnapshots;

        this.waitingQueue  = new ArrayDeque<>(queue);
        this.inHallway     = new ArrayList<>();
        this.atEntrance    = new ArrayList<>();
        this.atService     = new ArrayList<>();
        this.movingToSeat  = new ArrayList<>();
        this.sittingDown   = new ArrayList<>();
        this.seated        = new ArrayList<>();

        this.seatOccupied  = new boolean[cfg.tableRows][cfg.tableCols];
        this.snapshots     = new ArrayList<>();
    }

    public int run() {
        while (!isFinished()) tick();
        return totalTicks;
    }

    public void tick() {
        totalTicks++;
        boolean anyBlocked = false;
        justSeated.clear();

        // 앞 단계부터 처리
        anyBlocked |= processGroup(sittingDown);
        anyBlocked |= processGroup(movingToSeat);
        anyBlocked |= processGroup(atService);
        anyBlocked |= processGroup(atEntrance);
        anyBlocked |= processGroup(inHallway);

        // 착석 완료 처리 (ConcurrentModification 방지: tick 끝에 일괄 처리)
        for (Student s : justSeated) {
            sittingDown.remove(s);
            seatOccupied[s.getTargetRow()][s.getTargetCol()] = true;
            seated.add(s);
        }

        // 대기 큐 → 복도 진입
        int batchSize = Math.max(1, cfg.studentCount / 15);
        int entered = 0;
        while (!waitingQueue.isEmpty() && entered < batchSize) {
            Student s = waitingQueue.poll();
            s.setState(Student.State.HALLWAY_MOVE);
            s.initTicks(cfg);
            inHallway.add(s);
            entered++;
        }

        if (anyBlocked) bottleneckTicks++;
        if (recordSnapshots) snapshots.add(buildSnapshot());
    }

    private boolean processGroup(List<Student> group) {
        boolean blocked = false;
        List<Student> completed = new ArrayList<>();

        for (Student s : new ArrayList<>(group)) {
            if (s.tickDown()) {
                List<TickEvent> evts = TickEventRegistry.getSequence();
                int idx = s.getEventIndex();
                if (idx < evts.size()) {
                    evts.get(idx).onComplete(s, this);
                }
                completed.add(s);
                // 다음 단계로 이동
                promote(s);
            } else {
                blocked = true;
            }
        }
        group.removeAll(completed);
        return blocked;
    }

    private void promote(Student s) {
        switch (s.getState()) {
            case HALLWAY_MOVE  -> { s.advanceState(); s.initTicks(cfg); atEntrance.add(s); }
            case ENTRANCE_WAIT -> {
                s.advanceState(); s.initTicks(cfg);
                if (atService.size() < cfg.serviceLines * 3) atService.add(s);
                else atEntrance.add(s);
            }
            case FOOD_SERVICE  -> { s.advanceState(); s.initTicks(cfg); movingToSeat.add(s); }
            case SEAT_MOVE     -> { s.advanceState(); s.initTicks(cfg); sittingDown.add(s); }
            case SIT_DOWN      -> { /* onStudentSeated에서 처리 */ }
            default            -> {}
        }
    }

    public void onStudentSeated(Student s) {
        // tick 루프 중이므로 justSeated 버퍼에 추가, 실제 이동은 tick 끝에
        justSeated.add(s);
    }

    public boolean isFinished() {
        return waitingQueue.isEmpty()
            && inHallway.isEmpty()
            && atEntrance.isEmpty()
            && atService.isEmpty()
            && movingToSeat.isEmpty()
            && sittingDown.isEmpty();
    }

    private Map<String, Object> buildSnapshot() {
        Map<String, Object> snap = new LinkedHashMap<>();
        snap.put("tick",       totalTicks);
        snap.put("waiting",    waitingQueue.size());
        snap.put("hallway",    inHallway.size());
        snap.put("entrance",   atEntrance.size());
        snap.put("service",    atService.size());
        snap.put("movingSeat", movingToSeat.size());
        snap.put("sitting",    sittingDown.size());
        snap.put("seated",     seated.size());

        List<int[]> occupiedSeats = new ArrayList<>();
        for (int r = 0; r < cfg.tableRows; r++)
            for (int c = 0; c < cfg.tableCols; c++)
                if (seatOccupied[r][c]) occupiedSeats.add(new int[]{r, c});
        snap.put("seats", occupiedSeats);

        return snap;
    }

    public int getTotalTicks()      { return totalTicks; }
    public int getBottleneckTicks() { return bottleneckTicks; }
    public int getSeatedCount()     { return seated.size(); }
    public List<Map<String, Object>> getSnapshots() { return snapshots; }
}

JAVA_EOF

cat > cafeteria/StudentQueue.java << 'JAVA_EOF'
package cafeteria;

import java.util.*;

// ================================================================
// StudentQueue — 입장 전략별 학생 큐 생성
// ================================================================
public class StudentQueue {

    public static List<Student> create(CafeteriaConfig cfg) {
        return switch (cfg.strategy.toLowerCase()) {
            case "backtofront" -> backToFront(cfg);
            case "outsidein"   -> outsideIn(cfg);
            case "block"       -> blockMethod(cfg);
            case "steffen"     -> steffen(cfg);
            default            -> random(cfg);
        };
    }

    // ── 1. Random ────────────────────────────────────────────────
    private static List<Student> random(CafeteriaConfig cfg) {
        List<int[]> seats = allSeats(cfg);
        Collections.shuffle(seats);
        return toStudents(seats, cfg);
    }

    // ── 2. Back-to-Front (뒷줄부터) ─────────────────────────────
    private static List<Student> backToFront(CafeteriaConfig cfg) {
        List<int[]> seats = allSeats(cfg);
        seats.sort(Comparator.comparingInt((int[] s) -> s[0]).reversed());
        shuffleWithinRows(seats);
        return toStudents(seats, cfg);
    }

    // ── 3. Outside-In (창가→중간→통로) ──────────────────────────
    private static List<Student> outsideIn(CafeteriaConfig cfg) {
        List<int[]> seats = allSeats(cfg);
        int mid = cfg.tableCols / 2;
        // 창가(끝쪽) → 중간 순서
        seats.sort(Comparator.comparingInt((int[] s) ->
            Math.abs(s[1] - mid) < mid ? cfg.tableCols - Math.abs(s[1] - mid) : Math.abs(s[1] - mid)
        ));
        return toStudents(seats, cfg);
    }

    // ── 4. Block Method (3구역) ──────────────────────────────────
    private static List<Student> blockMethod(CafeteriaConfig cfg) {
        int zone = Math.max(1, cfg.tableRows / 3);
        List<int[]> back = new ArrayList<>(), mid = new ArrayList<>(), front = new ArrayList<>();
        for (int[] s : allSeats(cfg)) {
            if      (s[0] >= zone * 2) back.add(s);
            else if (s[0] >= zone)     mid.add(s);
            else                       front.add(s);
        }
        Collections.shuffle(back);
        Collections.shuffle(mid);
        Collections.shuffle(front);
        List<int[]> ordered = new ArrayList<>();
        ordered.addAll(back); ordered.addAll(mid); ordered.addAll(front);
        return toStudents(ordered, cfg);
    }

    // ── 5. Steffen (짝수→홀수 교차) ─────────────────────────────
    private static List<Student> steffen(CafeteriaConfig cfg) {
        List<int[]> seats = new ArrayList<>();
        int[] colOrder = buildColOrder(cfg.tableCols);
        for (int col : colOrder) {
            for (int row = cfg.tableRows - 2; row >= 0; row -= 2) seats.add(new int[]{row, col});
            for (int row = cfg.tableRows - 1; row >= 0; row -= 2) seats.add(new int[]{row, col});
        }
        return toStudents(seats, cfg);
    }

    // ── 공통 유틸 ────────────────────────────────────────────────

    private static List<int[]> allSeats(CafeteriaConfig cfg) {
        List<int[]> seats = new ArrayList<>();
        for (int r = 0; r < cfg.tableRows; r++)
            for (int c = 0; c < cfg.tableCols; c++)
                seats.add(new int[]{r, c});
        return seats;
    }

    private static List<Student> toStudents(List<int[]> seats, CafeteriaConfig cfg) {
        List<Student> list = new ArrayList<>();
        int total = Math.min(seats.size(), cfg.studentCount);
        for (int i = 0; i < total; i++)
            list.add(new Student(i, seats.get(i)[0], seats.get(i)[1], cfg));
        return list;
    }

    private static void shuffleWithinRows(List<int[]> seats) {
        Random rng = new Random();
        int i = 0;
        while (i < seats.size()) {
            int j = i, row = seats.get(i)[0];
            while (j < seats.size() && seats.get(j)[0] == row) j++;
            Collections.shuffle(seats.subList(i, j), rng);
            i = j;
        }
    }

    private static int[] buildColOrder(int cols) {
        // 창가(끝) → 중간 → 통로(가운데) 순서
        int[] order = new int[cols];
        int left = 0, right = cols - 1, idx = 0;
        while (left <= right) {
            if (left == right) order[idx++] = left++;
            else { order[idx++] = right--; order[idx++] = left++; }
        }
        return order;
    }
}

JAVA_EOF

cat > cafeteria/CafeteriaExporter.java << 'JAVA_EOF'
package cafeteria;

import java.util.*;
import java.io.*;

// ================================================================
// CafeteriaExporter — 시뮬레이션 전체를 실행하고 JSONL 저장
// ================================================================
public class CafeteriaExporter {

    public static void export(CafeteriaConfig cfg, String outFile) throws IOException {
        List<Student> queue = StudentQueue.create(cfg);
        CafeteriaSimulator sim = new CafeteriaSimulator(cfg, queue, true);
        sim.run();

        try (PrintWriter pw = new PrintWriter(new FileWriter(outFile))) {
            // 메타
            pw.printf("{\"meta\":{\"strategy\":\"%s\",\"students\":%d," +
                "\"tableRows\":%d,\"tableCols\":%d,\"lines\":%d," +
                "\"events\":[%s]}}%n",
                cfg.strategy, cfg.studentCount,
                cfg.tableRows, cfg.tableCols, cfg.serviceLines,
                String.join(",", TickEventRegistry.getNames().stream()
                    .map(n -> "\"" + n + "\"").toList())
            );

            // 스냅샷
            for (Map<String, Object> snap : sim.getSnapshots()) {
                pw.println(toJson(snap));
            }

            // 완료
            pw.printf("{\"done\":{\"total_ticks\":%d,\"seated\":%d,\"bottleneck\":%d}}%n",
                sim.getTotalTicks(), sim.getSeatedCount(), sim.getBottleneckTicks());
        }

        System.out.printf("Saved: %s  (strategy=%s, ticks=%d, seated=%d/%d)%n",
            outFile, cfg.strategy, sim.getTotalTicks(),
            sim.getSeatedCount(), cfg.studentCount);
    }

    @SuppressWarnings("unchecked")
    private static String toJson(Map<String, Object> map) {
        StringBuilder sb = new StringBuilder("{");
        boolean first = true;
        for (Map.Entry<String, Object> e : map.entrySet()) {
            if (!first) sb.append(",");
            sb.append("\"").append(e.getKey()).append("\":");
            Object v = e.getValue();
            if (v instanceof Integer || v instanceof Long) {
                sb.append(v);
            } else if (v instanceof List<?> list) {
                sb.append("[");
                for (int i = 0; i < list.size(); i++) {
                    if (i > 0) sb.append(",");
                    Object item = list.get(i);
                    if (item instanceof int[] arr) {
                        sb.append("[");
                        for (int j = 0; j < arr.length; j++) {
                            if (j > 0) sb.append(",");
                            sb.append(arr[j]);
                        }
                        sb.append("]");
                    } else {
                        sb.append(item);
                    }
                }
                sb.append("]");
            } else {
                sb.append(v);
            }
            first = false;
        }
        sb.append("}");
        return sb.toString();
    }

    public static void main(String[] args) throws IOException {
        String[] strategies = {"random", "backtofront", "outsidein", "block", "steffen"};
        CafeteriaConfig base = new CafeteriaConfig();

        for (String s : strategies) {
            CafeteriaConfig cfg = new CafeteriaConfig()
                .students(30).lines(2).tables(5, 6)
                .strategy(s);
            export(cfg, "cafeteria_" + s + ".jsonl");
        }
        System.out.println("All done.");
    }
}

JAVA_EOF

echo '✅ 자바 파일 6개 생성 완료'


✅ 자바 파일 6개 생성 완료


In [2]:
%%shell
javac cafeteria/*.java && echo '✅ 컴파일 성공'
echo ''
java -cp . cafeteria.CafeteriaExporter
echo ''
echo '--- JSONL 파일 생성 확인 ---'
ls -lh cafeteria_*.jsonl


✅ 컴파일 성공

Saved: cafeteria_random.jsonl  (strategy=random, ticks=32, seated=30/30)
Saved: cafeteria_backtofront.jsonl  (strategy=backtofront, ticks=30, seated=30/30)
Saved: cafeteria_outsidein.jsonl  (strategy=outsidein, ticks=32, seated=30/30)
Saved: cafeteria_block.jsonl  (strategy=block, ticks=29, seated=30/30)
Saved: cafeteria_steffen.jsonl  (strategy=steffen, ticks=29, seated=30/30)
All done.

--- JSONL 파일 생성 확인 ---
-rw-r--r-- 1 root root 5.0K Jun  5 06:53 cafeteria_backtofront.jsonl
-rw-r--r-- 1 root root 4.7K Jun  5 06:53 cafeteria_block.jsonl
-rw-r--r-- 1 root root 5.5K Jun  5 06:53 cafeteria_outsidein.jsonl
-rw-r--r-- 1 root root 5.5K Jun  5 06:53 cafeteria_random.jsonl
-rw-r--r-- 1 root root 4.7K Jun  5 06:53 cafeteria_steffen.jsonl


In [3]:
import json, subprocess, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ════════════════════════════════════════════════════════════════
# 설정 — 여기만 바꾸면 됩니다
# ════════════════════════════════════════════════════════════════
STRATEGY = "random"   # random | backtofront | outsidein | block | steffen
STUDENTS = 30
LINES    = 2          # 배식 라인 수
BAGGAGE  = 0.2        # 짐 있는 학생 비율 (0.0 ~ 1.0)

# ── Step 1: 자바로 데이터 생성 ────────────────────────────────────
JSONL = f"cafeteria_{STRATEGY}.jsonl"

cmd = [
    "java", "-cp", ".", "cafeteria.CafeteriaExporter"
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout.strip())

# ── Step 2: JSONL 파싱 ───────────────────────────────────────────
meta, frames, total_ticks = None, [], 0

with open(JSONL) as f:
    for line in f:
        obj = json.loads(line)
        if   "meta" in obj: meta = obj["meta"]
        elif "tick" in obj: frames.append(obj)
        elif "done" in obj: total_ticks = obj["done"]["total_ticks"]

ROWS  = meta["tableRows"]
COLS  = meta["tableCols"]
EVENTS = meta["events"]
print(f"Strategy: {meta['strategy']} | {len(frames)} frames | {total_ticks} ticks")
print(f"Events: {' → '.join(EVENTS)}")

# ════════════════════════════════════════════════════════════════
# 색상 팔레트
# ════════════════════════════════════════════════════════════════
COLORS = {
    "bg":        "#1a1a2e",
    "panel":     "#16213e",
    "empty":     "#2d2d44",
    "seated":    "#4CAF50",
    "waiting":   "#9C27B0",
    "hallway":   "#2196F3",
    "entrance":  "#00BCD4",
    "service":   "#FF9800",
    "moveSeat":  "#FF5722",
    "sitting":   "#F44336",
    "text":      "#e0e0e0",
    "subtext":   "#888888",
}

STATE_COLOR = {
    0: COLORS["waiting"],
    1: COLORS["hallway"],
    2: COLORS["entrance"],
    3: COLORS["service"],
    4: COLORS["moveSeat"],
    5: COLORS["sitting"],
    6: COLORS["seated"],
}

# ════════════════════════════════════════════════════════════════
# 레이아웃: 왼쪽=급식실 평면도, 오른쪽=단계별 현황 바
# ════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(14, 8), facecolor=COLORS["bg"])
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1], wspace=0.05)

ax_floor = fig.add_subplot(gs[0])   # 급식실 평면도
ax_bar   = fig.add_subplot(gs[1])   # 단계별 막대

for ax in [ax_floor, ax_bar]:
    ax.set_facecolor(COLORS["bg"])
    ax.axis("off")

# ── 평면도 설정 ──────────────────────────────────────────────────
PAD = 1.2
ax_floor.set_xlim(-PAD, COLS + PAD * 3)
ax_floor.set_ylim(-PAD * 2, ROWS + PAD * 2)
ax_floor.set_aspect("equal")

# 식탁 배경 (테이블 단위로 묶음)
for r in range(ROWS):
    table_bg = mpatches.FancyBboxPatch(
        (-0.1, r - 0.05), COLS + 0.2, 0.9,
        boxstyle="round,pad=0.05",
        facecolor=COLORS["panel"], edgecolor="#333355", linewidth=0.5, zorder=0
    )
    ax_floor.add_patch(table_bg)

# 좌석 사각형
seat_patches = {}
SW, SH = 0.82, 0.82
for r in range(ROWS):
    for c in range(COLS):
        patch = mpatches.FancyBboxPatch(
            (c + 0.09, r + 0.09), SW, SH,
            boxstyle="round,pad=0.04",
            facecolor=COLORS["empty"], edgecolor="#444466", linewidth=0.4, zorder=1
        )
        ax_floor.add_patch(patch)
        seat_patches[(r, c)] = patch

# 배식대 표시 (아래쪽)
service_rect = mpatches.FancyBboxPatch(
    (-0.1, -1.9), COLS + 0.2, 0.75,
    boxstyle="round,pad=0.05",
    facecolor="#1a3a2a", edgecolor="#4CAF50", linewidth=1.2, zorder=1
)
ax_floor.add_patch(service_rect)
ax_floor.text(COLS / 2, -1.55, "[ 배식대 ]",
    ha="center", va="center", fontsize=9, color="#4CAF50", fontweight="bold")

# 배식 라인 표시
for i in range(LINES):
    x = (i + 1) * COLS / (LINES + 1)
    line_circle = plt.Circle((x, -1.55), 0.22, color=COLORS["service"], zorder=2)
    ax_floor.add_patch(line_circle)
    ax_floor.text(x, -1.55, str(i+1), ha="center", va="center",
        fontsize=7, color="white", fontweight="bold", zorder=3)

# 열 레이블
for c in range(COLS):
    ax_floor.text(c + 0.5, ROWS + 0.35, chr(65 + c),
        ha="center", va="center", fontsize=8, color=COLORS["subtext"])

# 행 번호
for r in range(ROWS):
    ax_floor.text(-0.7, r + 0.5, str(r + 1),
        ha="center", va="center", fontsize=7, color=COLORS["subtext"])

# 복도 진입 화살표
ax_floor.annotate("", xy=(COLS / 2, -0.3), xytext=(COLS / 2, -1.1),
    arrowprops=dict(arrowstyle="->", color=COLORS["hallway"], lw=1.5))

# 제목
title_ax = ax_floor.text(
    COLS / 2, ROWS + 1.0, f"Cafeteria Boarding — {meta['strategy'].upper()}",
    ha="center", va="center", fontsize=12, fontweight="bold", color=COLORS["text"]
)
tick_info = ax_floor.text(
    COLS / 2, ROWS + 0.6, "tick: 0 | seated: 0/30",
    ha="center", va="center", fontsize=9, color=COLORS["subtext"]
)

# ── 단계별 현황 바 설정 ──────────────────────────────────────────
STAGE_LABELS = ["Waiting", "Hallway", "Entrance", "Service", "To Seat", "Sitting", "Seated"]
STAGE_COLORS = [
    COLORS["waiting"], COLORS["hallway"], COLORS["entrance"],
    COLORS["service"], COLORS["moveSeat"], COLORS["sitting"], COLORS["seated"]
]
STAGE_KEYS   = ["waiting", "hallway", "entrance", "service", "movingSeat", "sitting", "seated"]

ax_bar.set_xlim(0, 1)
ax_bar.set_ylim(-0.5, len(STAGE_LABELS) + 0.5)

bar_patches = {}
bar_texts   = {}
bar_count_texts = {}

for i, (label, color) in enumerate(zip(STAGE_LABELS, STAGE_COLORS)):
    y = len(STAGE_LABELS) - 1 - i
    # 배경 바
    bg = mpatches.FancyBboxPatch(
        (0.05, y + 0.1), 0.9, 0.7,
        boxstyle="round,pad=0.02",
        facecolor="#2a2a3e", edgecolor="#444466", linewidth=0.4, zorder=1
    )
    ax_bar.add_patch(bg)
    # 진행 바
    bar = mpatches.FancyBboxPatch(
        (0.05, y + 0.1), 0.001, 0.7,
        boxstyle="round,pad=0.02",
        facecolor=color, edgecolor="none", linewidth=0, zorder=2
    )
    ax_bar.add_patch(bar)
    bar_patches[STAGE_KEYS[i]] = bar

    # 레이블
    ax_bar.text(0.08, y + 0.45, label,
        ha="left", va="center", fontsize=8, color=COLORS["text"], zorder=3)
    # 숫자
    cnt = ax_bar.text(0.92, y + 0.45, "0",
        ha="right", va="center", fontsize=8, color=COLORS["text"], zorder=3)
    bar_count_texts[STAGE_KEYS[i]] = cnt

ax_bar.text(0.5, len(STAGE_LABELS) + 0.2, "Stage Progress",
    ha="center", va="center", fontsize=9, color=COLORS["text"], fontweight="bold")

# ── 범례 ─────────────────────────────────────────────────────────
legend_items = [mpatches.Patch(color=c, label=l)
    for l, c in zip(STAGE_LABELS, STAGE_COLORS)]
ax_floor.legend(handles=legend_items, loc="lower left",
    bbox_to_anchor=(-0.05, -0.28), ncol=4, fontsize=7,
    facecolor=COLORS["panel"], edgecolor="#444466", labelcolor=COLORS["text"])

# ════════════════════════════════════════════════════════════════
# 애니메이션 업데이트 함수
# ════════════════════════════════════════════════════════════════
prev_seated = set()

def update(fi):
    global prev_seated
    if fi >= len(frames): return

    frame = frames[fi]

    # 새로 착석된 좌석 색상 업데이트
    current_seated = set(tuple(s) for s in frame["seats"])
    for (r, c) in current_seated - prev_seated:
        seat_patches[(r, c)].set_facecolor(COLORS["seated"])
        seat_patches[(r, c)].set_edgecolor("#2e7d32")
    prev_seated = current_seated

    # tick 정보 업데이트
    tick_info.set_text(
        f"tick: {frame['tick']}  |  seated: {frame['seated']}/{STUDENTS}"
    )

    # 단계별 막대 업데이트
    total = max(STUDENTS, 1)
    for key in STAGE_KEYS:
        val = frame.get(key, 0)
        ratio = min(val / total, 1.0)
        bar_patches[key].set_width(max(0.001, ratio * 0.9))
        bar_count_texts[key].set_text(str(val))

plt.tight_layout(pad=0.5)

# ── 프레임 다운샘플 ──────────────────────────────────────────────
SKIP    = max(1, len(frames) // 120)
sampled = list(range(0, len(frames), SKIP)) + [len(frames) - 1]

anim = FuncAnimation(
    fig, update,
    frames=sampled,
    interval=120,
    blit=False,
    repeat=False
)

plt.close()
HTML(anim.to_jshtml())



Saved: cafeteria_random.jsonl  (strategy=random, ticks=32, seated=30/30)
Saved: cafeteria_backtofront.jsonl  (strategy=backtofront, ticks=30, seated=30/30)
Saved: cafeteria_outsidein.jsonl  (strategy=outsidein, ticks=31, seated=30/30)
Saved: cafeteria_block.jsonl  (strategy=block, ticks=32, seated=30/30)
Saved: cafeteria_steffen.jsonl  (strategy=steffen, ticks=29, seated=30/30)
All done.
Strategy: random | 32 frames | 32 ticks
Events: HALLWAY_MOVE → ENTRANCE_WAIT → FOOD_SERVICE → SEAT_MOVE → SIT_DOWN


/tmp/ipykernel_621/2041866745.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0.5)
/tmp/ipykernel_621/2041866745.py:250: UserWarning: Glyph 48176 (\N{HANGUL SYLLABLE BAE}) missing from font(s) DejaVu Sans.
  HTML(anim.to_jshtml())
/tmp/ipykernel_621/2041866745.py:250: UserWarning: Glyph 49885 (\N{HANGUL SYLLABLE SIG}) missing from font(s) DejaVu Sans.
  HTML(anim.to_jshtml())
/tmp/ipykernel_621/2041866745.py:250: UserWarning: Glyph 45824 (\N{HANGUL SYLLABLE DAE}) missing from font(s) DejaVu Sans.
  HTML(anim.to_jshtml())


## 커스텀 이벤트 추가 방법

실험자가 새 병목 상황을 추가하려면 자바 파일에 아래 코드를 추가하고 재컴파일하세요.

```java
// 예시 1: 배식대 앞에서 카드 결제 이벤트 추가
class PaymentEvent implements TickEvent {
    @Override public String name() { return "PAYMENT"; }
    @Override public int durationTicks(Student s, CafeteriaConfig cfg) {
        return 3; // 결제 3tick
    }
}
// FOOD_SERVICE 이후에 삽입
TickEventRegistry.insertAfter("FOOD_SERVICE", new PaymentEvent());

// 예시 2: 배식 속도를 절반으로 줄이기
class SlowServiceEvent implements TickEvent {
    @Override public String name() { return "FOOD_SERVICE"; }
    @Override public int durationTicks(Student s, CafeteriaConfig cfg) {
        return cfg.menuItems * cfg.serviceTimePerItem * 2; // 2배 느림
    }
}
TickEventRegistry.replace("FOOD_SERVICE", new SlowServiceEvent());
```

**변수 실험 조합 추천:**

| 실험 | STRATEGY | LINES | STUDENTS | 목적 |
|---|---|---|---|---|
| 기본 | `random` | 2 | 30 | 기준값 |
| 배식 라인 늘리기 | `random` | 4 | 30 | 라인 증가 효과 |
| 학생 수 증가 | `random` | 2 | 60 | 과밀 상황 |
| 최적 전략 비교 | `outsidein` | 2 | 30 | 최고 전략 확인 |
| 최악 전략 확인 | `backtofront` | 2 | 30 | 병목 시각화 |
